# Cognix Phase 3: Inference on Provo eye-tracking corpus

**Purpose:** Run TRIBE v2, LLaMA 3.2-3B, and sentence-transformer on all 55 Provo passages. Cache vectors to Google Drive. The analysis notebook (`04_phase3_analysis.ipynb`) then loads these cached vectors locally and tests whether Cognix predicts reading time better than the baselines.

**Setup:**
1. Add HuggingFace token as Colab secret (`HF_TOKEN`) — needs LLaMA 3.2 access.
2. Runtime > Change runtime type > **A100 GPU + High RAM** (L4 also works, slower).
3. Run all cells.
4. Total inference time: ~35 min TRIBE + <1 min LLaMA + <1 min sentence-transformer.

**Resumable:** All loops skip already-cached vectors. Re-run after a Colab disconnect to pick up where it left off.

**Output cache structure on Drive:**
```
/content/drive/MyDrive/cognix_cache/
  phase3_provo_pooled/   # TRIBE pooled (20484-d), one .npy per passage
  phase3_provo_llama/    # LLaMA 3.2-3B mean-pooled (3072-d)
  phase3_provo_st/       # sentence-transformer (384-d, all-MiniLM-L6-v2)
```

In [ ]:
# ---- System dependencies ----
!apt-get update -qq && apt-get install -y -qq ffmpeg

# ---- Install dependencies (mirrors Round 2 setup) ----
!pip uninstall -y numpy
!pip install 'numpy>=1.26.4,<2.1.0'

!git clone https://github.com/facebookresearch/tribev2.git /content/tribev2-repo
!pip install -e '/content/tribev2-repo/.[plotting]'
!pip install torchcodec==0.2.1 sentence-transformers tqdm

# ---- Clone Cognix repo ----
!git clone https://github.com/gdavid7/cognix.git /content/cognix 2>/dev/null || (cd /content/cognix && git pull)

# ---- HuggingFace auth ----
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# NOTE: Colab may ask to restart after the numpy install. Click Restart,
# then SKIP this cell and run from the next cell onward.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CACHE_ROOT = '/content/drive/MyDrive/cognix_cache'
POOLED_DIR = f'{CACHE_ROOT}/phase3_provo_pooled'
LLAMA_DIR  = f'{CACHE_ROOT}/phase3_provo_llama'
ST_DIR     = f'{CACHE_ROOT}/phase3_provo_st'

for d in [POOLED_DIR, LLAMA_DIR, ST_DIR]:
    os.makedirs(d, exist_ok=True)
print('Cache directories ready:')
print(f'  TRIBE pooled: {POOLED_DIR}')
print(f'  LLaMA:        {LLAMA_DIR}')
print(f'  ST:           {ST_DIR}')

In [ ]:
import json
import time
import tempfile
import numpy as np
from pathlib import Path
from tqdm import tqdm

## 1. Load Provo chunks

These were preprocessed locally by `cognix.provo_loader` and committed to `data/phase3_provo_chunks.jsonl`. Each entry has the passage text, a stable hash (SHA-256[:16]), and aggregated reading-time targets.

In [ ]:
CHUNKS_PATH = '/content/cognix/data/phase3_provo_chunks.jsonl'

chunks = []
with open(CHUNKS_PATH) as f:
    for line in f:
        chunks.append(json.loads(line))
print(f'Loaded {len(chunks)} Provo chunks')
print(f'Word count range: {min(c["n_words"] for c in chunks)}-{max(c["n_words"] for c in chunks)}')

# Sanity-check that hashes are unique (no duplicate passages)
hashes = [c['hash'] for c in chunks]
assert len(set(hashes)) == len(hashes), 'Duplicate hashes found in chunks!'
print(f'All {len(hashes)} hashes unique. ✓')

In [ ]:
# ---- Check what's already cached for each model ----
def already_cached(d):
    return {p.stem for p in Path(d).glob('*.npy')}

tribe_done = already_cached(POOLED_DIR)
llama_done = already_cached(LLAMA_DIR)
st_done    = already_cached(ST_DIR)

tribe_remaining = [c for c in chunks if c['hash'] not in tribe_done]
llama_remaining = [c for c in chunks if c['hash'] not in llama_done]
st_remaining    = [c for c in chunks if c['hash'] not in st_done]

print(f'TRIBE: {len(tribe_done)} done, {len(tribe_remaining)} remaining (~{len(tribe_remaining)*38/60:.0f} min on A100)')
print(f'LLaMA: {len(llama_done)} done, {len(llama_remaining)} remaining')
print(f'ST:    {len(st_done)} done, {len(st_remaining)} remaining')

## 2. TRIBE v2 inference

Same wrapper pattern as Round 2 (`01_r2_tribe_inference.ipynb`). Mean-pools across the time dimension to get one (20484,) vector per passage.

In [ ]:
from tribev2 import TribeModel
tribe_model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='./cache')
print('TRIBE v2 loaded.')

In [ ]:
def text_to_pooled_brain(text):
    """Run TRIBE on text, return mean-pooled brain vector (20484,)."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
        tmp_path = f.name
    try:
        events = tribe_model.get_events_dataframe(text_path=tmp_path)
        preds, _segments = tribe_model.predict(events=events)
        return np.asarray(preds).mean(axis=0)  # (T, 20484) -> (20484,)
    finally:
        os.unlink(tmp_path)

In [ ]:
# ---- TRIBE inference loop (resumable) ----
tribe_failed = []
start = time.time()

for c in tqdm(tribe_remaining, desc='TRIBE inference'):
    out_path = Path(POOLED_DIR) / f"{c['hash']}.npy"
    if out_path.exists():
        continue
    try:
        vec = text_to_pooled_brain(c['text'])
        assert vec.shape == (20484,), f'Bad shape: {vec.shape}'
        np.save(out_path, vec)
    except Exception as e:
        tribe_failed.append({'text_id': c['text_id'], 'hash': c['hash'], 'error': str(e)})
        print(f"FAILED text_id={c['text_id']} hash={c['hash']}: {e}")

elapsed = time.time() - start
print(f'\nDone. Processed {len(tribe_remaining) - len(tribe_failed)} passages in {elapsed/60:.1f} min')
print(f'Failed: {len(tribe_failed)}')
for f_entry in tribe_failed:
    print(f"  text_id={f_entry['text_id']}: {f_entry['error']}")

In [ ]:
# ---- Verify TRIBE outputs ----
n_cached = len(list(Path(POOLED_DIR).glob('*.npy')))
print(f'TRIBE vectors on Drive: {n_cached}/{len(chunks)}')

# Spot-check first chunk: shape, dtype, no NaNs/Infs
sample = np.load(Path(POOLED_DIR) / f"{chunks[0]['hash']}.npy")
assert sample.shape == (20484,), f'Bad shape: {sample.shape}'
assert not np.isnan(sample).any(), 'NaN in TRIBE output'
assert not np.isinf(sample).any(), 'Inf in TRIBE output'
print(f'Sample shape: {sample.shape}, dtype: {sample.dtype}, mean: {sample.mean():.4f}, std: {sample.std():.4f}')
print('TRIBE outputs look healthy. ✓')

## 3. LLaMA 3.2-3B embeddings

Same model TRIBE uses internally for text features. Mean-pool the last hidden state across tokens to get one (3072,) vector per passage.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

llama_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B')
llama_model = AutoModel.from_pretrained('meta-llama/Llama-3.2-3B', torch_dtype=torch.float16)
llama_model = llama_model.cuda().eval()
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token
print(f'LLaMA 3.2-3B loaded. Hidden size: {llama_model.config.hidden_size}')

In [ ]:
llama_failed = []
for c in tqdm(llama_remaining, desc='LLaMA embeddings'):
    out_path = Path(LLAMA_DIR) / f"{c['hash']}.npy"
    if out_path.exists():
        continue
    try:
        inputs = llama_tokenizer(
            c['text'], return_tensors='pt', truncation=True, max_length=512, padding=False
        ).to('cuda')
        with torch.no_grad():
            outputs = llama_model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
        assert emb.shape == (3072,), f'Bad shape: {emb.shape}'
        np.save(out_path, emb)
    except Exception as e:
        llama_failed.append({'text_id': c['text_id'], 'error': str(e)})
        print(f"FAILED text_id={c['text_id']}: {e}")

n_llama = len(list(Path(LLAMA_DIR).glob('*.npy')))
print(f'\nLLaMA vectors on Drive: {n_llama}/{len(chunks)}')
if llama_failed:
    print(f'Failed: {len(llama_failed)}')

# Spot check
sample = np.load(Path(LLAMA_DIR) / f"{chunks[0]['hash']}.npy")
assert not np.isnan(sample).any() and not np.isinf(sample).any()
print(f'Sample shape: {sample.shape}, dtype: {sample.dtype}, mean: {sample.mean():.4f}')
print('LLaMA outputs look healthy. ✓')

## 4. Sentence-transformer embeddings

Lightweight semantic baseline (`all-MiniLM-L6-v2`, 384-d). Used in Round 2.

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Sentence-transformer loaded.')

In [ ]:
st_failed = []
for c in tqdm(st_remaining, desc='ST embeddings'):
    out_path = Path(ST_DIR) / f"{c['hash']}.npy"
    if out_path.exists():
        continue
    try:
        emb = st_model.encode(c['text'])
        assert emb.shape == (384,), f'Bad shape: {emb.shape}'
        np.save(out_path, emb)
    except Exception as e:
        st_failed.append({'text_id': c['text_id'], 'error': str(e)})
        print(f"FAILED text_id={c['text_id']}: {e}")

n_st = len(list(Path(ST_DIR).glob('*.npy')))
print(f'\nST vectors on Drive: {n_st}/{len(chunks)}')

sample = np.load(Path(ST_DIR) / f"{chunks[0]['hash']}.npy")
assert not np.isnan(sample).any() and not np.isinf(sample).any()
print(f'Sample shape: {sample.shape}')
print('ST outputs look healthy. ✓')

## 5. Final summary

All three feature sources should now be cached on Drive. Download the three directories to the local Mac for analysis in `04_phase3_analysis.ipynb`.

In [ ]:
n_tribe = len(list(Path(POOLED_DIR).glob('*.npy')))
n_llama = len(list(Path(LLAMA_DIR).glob('*.npy')))
n_st    = len(list(Path(ST_DIR).glob('*.npy')))
n_target = len(chunks)

print('PHASE 3 INFERENCE SUMMARY')
print('=' * 50)
print(f'Target passages:   {n_target}')
print(f'TRIBE vectors:     {n_tribe}/{n_target} ({100*n_tribe/n_target:.0f}%)')
print(f'LLaMA vectors:     {n_llama}/{n_target} ({100*n_llama/n_target:.0f}%)')
print(f'ST vectors:        {n_st}/{n_target} ({100*n_st/n_target:.0f}%)')
print()
if n_tribe == n_llama == n_st == n_target:
    print('ALL COMPLETE. Ready for analysis.')
else:
    print('INCOMPLETE. Re-run this notebook to retry failed passages.')
print()
print('Download these directories from Drive to local Mac:')
print(f'  {POOLED_DIR}')
print(f'  {LLAMA_DIR}')
print(f'  {ST_DIR}')